# Strategic Market Visualization – Simulated Antique Cartography Dataset

This notebook transforms a public retail transaction dataset into a simulated antique map and print market dataset for Tableau analysis.

In [1]:
import pandas as pd
import numpy as np

## Load the dataset

In [2]:
from google.colab import files
uploaded = files.upload()

Saving online_retail.csv to online_retail.csv


In [4]:
df = pd.read_csv("online_retail.csv", encoding="ISO-8859-1")
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## Check the shape and columns

In [5]:
print(df.shape)
print(df.columns.tolist())

(541909, 8)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


## Preview the main fields we may use for the simulation

In [6]:
df[["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "Country"]].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,United Kingdom


## Check missing values in the main fields

In [7]:
print(df[["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "Country"]].isnull().sum())

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
Country           0
dtype: int64


# Step 2: Clean the base dataset

In this step, we keep only the core transaction columns, remove rows with missing values, remove cancelled invoices, and keep only positive quantity and price values.

## Keep only the columns needed for the simulation

In [8]:
columns_needed = [
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "Country"
]

df = df[columns_needed].copy()
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,United Kingdom


## Remove rows with missing values in the main fields

In [9]:
df = df.dropna(subset=[
    "InvoiceNo",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "UnitPrice",
    "Country"
]).copy()

print(df.shape)

(540455, 7)


## Convert InvoiceDate to datetime

In [10]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print(df["InvoiceDate"].dtype)

datetime64[ns]


## Remove cancelled invoices

Cancelled invoices usually start with the letter C.

In [11]:
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")].copy()
print(df.shape)

(531167, 7)


## Keep only positive quantity and positive price rows

In [12]:
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
print(df.shape)

(530104, 7)


## Standardize text fields

In [13]:
df["Description"] = df["Description"].astype(str).str.strip()
df["Country"] = df["Country"].astype(str).str.strip()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,United Kingdom


## Create a base sale price field

This gives us a direct price field to reuse in the auction-style transformation.

In [14]:
df["Sale_Price"] = df["UnitPrice"] * df["Quantity"]
df[["Quantity", "UnitPrice", "Sale_Price"]].head()

,Quantity,UnitPrice,Sale_Price
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34


## Check the cleaned dataset

In [15]:
print(df.shape)
print(df.head())

(530104, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice         Country  Sale_Price  
0 2010-12-01 08:26:00       2.55  United Kingdom       15.30  
1 2010-12-01 08:26:00       3.39  United Kingdom       20.34  
2 2010-12-01 08:26:00       2.75  United Kingdom       22.00  
3 2010-12-01 08:26:00       3.39  United Kingdom       20.34  
4 2010-12-01 08:26:00       3.39  United Kingdom       20.34  


# Step 3: Create the simulated antique cartography market fields

In this step, the cleaned retail transaction dataset is transformed into a simulated antique map and print market dataset. New fields are created to represent century, cartographer, material, provenance region, rarity, auction activity, and acquisition scoring logic.

## Create a simulated artifact title field

We use the original product description as the base object title.

In [16]:
df["Artifact_Title"] = df["Description"]
df[["Description", "Artifact_Title"]].head()

,Description,Artifact_Title
0,WHITE HANGING HEART T-LIGHT HOLDER,WHITE HANGING HEART T-LIGHT HOLDER
1,WHITE METAL LANTERN,WHITE METAL LANTERN
2,CREAM CUPID HEARTS COAT HANGER,CREAM CUPID HEARTS COAT HANGER
3,KNITTED UNION FLAG HOT WATER BOTTLE,KNITTED UNION FLAG HOT WATER BOTTLE
4,RED WOOLLY HOTTIE WHITE HEART.,RED WOOLLY HOTTIE WHITE HEART.


## Create a simulated sale date field

In [17]:
df["Sale_Date"] = df["InvoiceDate"]
df[["InvoiceDate", "Sale_Date"]].head()

,InvoiceDate,Sale_Date
0,2010-12-01 08:26:00,2010-12-01 08:26:00
1,2010-12-01 08:26:00,2010-12-01 08:26:00
2,2010-12-01 08:26:00,2010-12-01 08:26:00
3,2010-12-01 08:26:00,2010-12-01 08:26:00
4,2010-12-01 08:26:00,2010-12-01 08:26:00


## Create a simulated cartographer field

This creates a rotating list of historic-style cartographer labels for the simulated market.

In [18]:
cartographers = [
    "Mercator Studio",
    "Blaeu Workshop",
    "Ortelius Circle",
    "Visscher Atelier",
    "Hondius Press",
    "Coronelli Archive",
    "Homann Engraving House",
    "De Wit Collection"
]

df["Cartographer"] = [cartographers[i % len(cartographers)] for i in range(len(df))]
df[["Artifact_Title", "Cartographer"]].head()

,Artifact_Title,Cartographer
0,WHITE HANGING HEART T-LIGHT HOLDER,Mercator Studio
1,WHITE METAL LANTERN,Blaeu Workshop
2,CREAM CUPID HEARTS COAT HANGER,Ortelius Circle
3,KNITTED UNION FLAG HOT WATER BOTTLE,Visscher Atelier
4,RED WOOLLY HOTTIE WHITE HEART.,Hondius Press


## Create a century field from the sale date

This is a simulation field for comparing historical eras in the dashboard.

In [19]:
centuries = ["16th Century", "17th Century", "18th Century", "19th Century"]

df["Century"] = [centuries[i % len(centuries)] for i in range(len(df))]
df["Century"].value_counts()

,count
Century,
16th Century,132526
17th Century,132526
18th Century,132526
19th Century,132526


## Create a material group field

This simulates antique map and print materials based on title keywords and rotation logic.

In [20]:
def map_material(title, idx):
    t = str(title).lower()
    if "paper" in t or "card" in t:
        return "Paper Print"
    elif "glass" in t:
        return "Hand-Colored Engraving"
    elif "wood" in t:
        return "Mounted Atlas Board"
    else:
        materials = ["Vellum", "Hand-Colored Engraving", "Paper Print", "Mounted Atlas Board"]
        return materials[idx % len(materials)]

df["Material_Group"] = [map_material(title, i) for i, title in enumerate(df["Artifact_Title"])]
df["Material_Group"].value_counts()

,count
Material_Group,
Paper Print,147627
Mounted Atlas Board,140412
Hand-Colored Engraving,127433
Vellum,114632


## Create a provenance region field from country

This repurposes the country field into a broader provenance/sale-region grouping.

In [21]:
def map_region(country):
    c = str(country).lower()
    if c in ["united kingdom", "eire", "france", "germany", "netherlands", "belgium", "spain", "italy", "switzerland", "portugal", "norway", "sweden", "finland", "denmark", "austria", "cyprus", "malta", "poland", "czech republic", "lithuania", "iceland", "greece", "channel islands", "european community"]:
        return "Europe"
    elif c in ["usa", "united states", "canada"]:
        return "North America"
    elif c in ["australia", "japan", "singapore", "hong kong", "israel", "lebanon", "bahrain", "saudi arabia", "united arab emirates"]:
        return "Asia-Pacific & Middle East"
    else:
        return "International"

df["Provenance_Region"] = df["Country"].apply(map_region)
df["Provenance_Region"].value_counts()

,count
Provenance_Region,
Europe,526795
Asia-Pacific & Middle East,2444
International,535
North America,330


## Create auction frequency by artifact title

This acts as a proxy for how often a specimen appears in the market.

In [23]:
auction_freq = df.groupby("Artifact_Title").size().reset_index(name="Auction_Frequency")
df = df.merge(auction_freq, on="Artifact_Title", how="left")
df[["Artifact_Title", "Auction_Frequency"]].head()

,Artifact_Title,Auction_Frequency
0,WHITE HANGING HEART T-LIGHT HOLDER,2323
1,WHITE METAL LANTERN,319
2,CREAM CUPID HEARTS COAT HANGER,287
3,KNITTED UNION FLAG HOT WATER BOTTLE,469
4,RED WOOLLY HOTTIE WHITE HEART.,443


## Create a rarity score

Lower market frequency means higher rarity.

In [24]:
df["Rarity_Score"] = (1 / df["Auction_Frequency"]) * 100
df["Rarity_Score"] = df["Rarity_Score"].round(2)
df[["Auction_Frequency", "Rarity_Score"]].head()

,Auction_Frequency,Rarity_Score
0,2323,0.04
1,319,0.31
2,287,0.35
3,469,0.21
4,443,0.23


## Create a material prestige score

This gives a simple value weight to different simulated materials.

In [25]:
prestige_map = {
    "Vellum": 4,
    "Hand-Colored Engraving": 3,
    "Mounted Atlas Board": 2,
    "Paper Print": 1
}

df["Material_Prestige"] = df["Material_Group"].map(prestige_map).fillna(1)
df[["Material_Group", "Material_Prestige"]].drop_duplicates()

,Material_Group,Material_Prestige
0,Vellum,4
1,Hand-Colored Engraving,3
2,Paper Print,1
3,Mounted Atlas Board,2


## Create a century preference score

This simulates collector preference by era.

In [26]:
century_map = {
    "16th Century": 4,
    "17th Century": 3,
    "18th Century": 2,
    "19th Century": 1
}

df["Century_Preference"] = df["Century"].map(century_map).fillna(1)
df[["Century", "Century_Preference"]].drop_duplicates()

,Century,Century_Preference
0,16th Century,4
1,17th Century,3
2,18th Century,2
3,19th Century,1


## Create acquisition priority score

This combines rarity, sale price, material prestige, and century preference into one acquisition-focused score.

In [27]:
sale_scaled = np.log1p(df["Sale_Price"])

df["Acquisition_Priority_Score"] = (
    df["Rarity_Score"] * 0.35 +
    sale_scaled * 0.20 +
    df["Material_Prestige"] * 10 * 0.20 +
    df["Century_Preference"] * 10 * 0.25
).round(2)

df[[
    "Rarity_Score",
    "Sale_Price",
    "Material_Prestige",
    "Century_Preference",
    "Acquisition_Priority_Score"
]].head()

,Rarity_Score,Sale_Price,Material_Prestige,Century_Preference,Acquisition_Priority_Score
0,0.04,15.30,4,4,18.57
1,0.31,20.34,3,3,14.22
2,0.35,22.00,1,2,7.75
3,0.21,20.34,2,1,7.19
4,0.23,20.34,4,4,18.69


## Preview the simulated antique market dataset

In [28]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country,Sale_Price,Artifact_Title,Sale_Date,Cartographer,Century,Material_Group,Provenance_Region,Auction_Frequency,Rarity_Score,Material_Prestige,Century_Preference,Acquisition_Priority_Score
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,United Kingdom,15.30,WHITE HANGING HEART T-LIGHT HOLDER,2010-12-01 08:26:00,Mercator Studio,16th Century,Vellum,Europe,2323,0.04,4,4,18.57
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,United Kingdom,20.34,WHITE METAL LANTERN,2010-12-01 08:26:00,Blaeu Workshop,17th Century,Hand-Colored Engraving,Europe,319,0.31,3,3,14.22
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,United Kingdom,22.00,CREAM CUPID HEARTS COAT HANGER,2010-12-01 08:26:00,Ortelius Circle,18th Century,Paper Print,Europe,287,0.35,1,2,7.75
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,United Kingdom,20.34,KNITTED UNION FLAG HOT WATER BOTTLE,2010-12-01 08:26:00,Visscher Atelier,19th Century,Mounted Atlas Board,Europe,469,0.21,2,1,7.19
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,United Kingdom,20.34,RED WOOLLY HOTTIE WHITE HEART.,2010-12-01 08:26:00,Hondius Press,16th Century,Vellum,Europe,443,0.23,4,4,18.69


# Step 4: Create the final Tableau-ready antique market dataset

In this step, we keep only the final simulation fields needed for Tableau and save the transformed dataset as a new CSV file.

## Keep only the final columns needed for analysis

In [29]:
final_columns = [
    "StockCode",
    "Artifact_Title",
    "Cartographer",
    "Century",
    "Material_Group",
    "Provenance_Region",
    "Sale_Date",
    "Sale_Price",
    "Auction_Frequency",
    "Rarity_Score",
    "Material_Prestige",
    "Century_Preference",
    "Acquisition_Priority_Score",
    "Country"
]

final_df = df[final_columns].copy()
final_df.head()

,StockCode,Artifact_Title,Cartographer,Century,Material_Group,Provenance_Region,Sale_Date,Sale_Price,Auction_Frequency,Rarity_Score,Material_Prestige,Century_Preference,Acquisition_Priority_Score,Country
0,85123A,WHITE HANGING HEART T-LIGHT HOLDER,Mercator Studio,16th Century,Vellum,Europe,2010-12-01 08:26:00,15.30,2323,0.04,4,4,18.57,United Kingdom
1,71053,WHITE METAL LANTERN,Blaeu Workshop,17th Century,Hand-Colored Engraving,Europe,2010-12-01 08:26:00,20.34,319,0.31,3,3,14.22,United Kingdom
2,84406B,CREAM CUPID HEARTS COAT HANGER,Ortelius Circle,18th Century,Paper Print,Europe,2010-12-01 08:26:00,22.00,287,0.35,1,2,7.75,United Kingdom
3,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,Visscher Atelier,19th Century,Mounted Atlas Board,Europe,2010-12-01 08:26:00,20.34,469,0.21,2,1,7.19,United Kingdom
4,84029E,RED WOOLLY HOTTIE WHITE HEART.,Hondius Press,16th Century,Vellum,Europe,2010-12-01 08:26:00,20.34,443,0.23,4,4,18.69,United Kingdom


## Check the final shape and columns

In [30]:
print(final_df.shape)
print(final_df.columns.tolist())

(530104, 14)
['StockCode', 'Artifact_Title', 'Cartographer', 'Century', 'Material_Group', 'Provenance_Region', 'Sale_Date', 'Sale_Price', 'Auction_Frequency', 'Rarity_Score', 'Material_Prestige', 'Century_Preference', 'Acquisition_Priority_Score', 'Country']


## Preview summary statistics for the key numeric fields

In [31]:
final_df[[
    "Sale_Price",
    "Auction_Frequency",
    "Rarity_Score",
    "Acquisition_Priority_Score"
]].describe()

,Sale_Price,Auction_Frequency,Rarity_Score,Acquisition_Priority_Score
count,530104.000000,530104.000000,530104.000000,530104.000000
mean,20.121871,417.337711,0.757479,11.770105
std,270.356743,380.897706,2.662394,4.723322
min,0.001000,1.000000,0.040000,4.630000
25%,3.750000,149.000000,0.180000,7.410000
50%,9.900000,300.000000,0.330000,10.150000
75%,17.700000,555.000000,0.670000,14.590000
max,168469.600000,2323.000000,100.000000,53.830000


## Save the final Tableau-ready dataset

In [32]:
final_df.to_csv("simulated_antique_cartography_market.csv", index=False)
print("Saved as simulated_antique_cartography_market.csv")

Saved as simulated_antique_cartography_market.csv


In [33]:
from google.colab import files
files.download("simulated_antique_cartography_market.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>